# 01 — Download the images

Downloads the metadata and images of the **Fons Martí Massafont Costals** collection via the Europeana API.

The collection contains 9,021 digitised photographs taken by commercial street photographer Martí Massafont Costals (1918–2012) in and around Girona between 1947 and 1997. The photographs are held by the Centre for Image Research and Dissemination (CRDI) of Girona City Council and made publicly available through Europeana under a CC BY-NC-ND 4.0 licence.

**Before running this notebook:**
- Register for a free Europeana API key at https://apis.europeana.eu/api/key-manager
- Fill in your API key in the Configuration cell below

**Outputs:**
- `massafont_metadata.csv` — metadata for all 9,021 records
- `images/massafont/` — folder of downloaded JPEG images (~800KB each, ~7GB total)

**Runtime:** Approximately 2–3 hours on a standard internet connection.

In [ ]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

## Configuration

Set your Europeana API key and output paths here. All other cells use these variables.

In [ ]:
# ── Configuration — edit these values ─────────────────────────────────────────
API_KEY        = "YOUR_EUROPEANA_API_KEY"   # Free key: https://apis.europeana.eu/api/key-manager
OUTPUT_FOLDER  = "images/massafont"         # Where to save downloaded images
METADATA_CSV   = "massafont_metadata.csv"   # Where to save the metadata

# ── Fixed collection identifiers — do not change ──────────────────────────────
COLLECTION     = "2024914_EuropeanaPhotography_Girona_1019"
FONDS          = "Fons Martí Massafont Costals"
PAGE_SIZE      = 100      # Maximum allowed by the Europeana API
DELAY_S        = 0.5      # Pause between image downloads (be polite to the server)
# ──────────────────────────────────────────────────────────────────────────────

Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"Metadata CSV:  {METADATA_CSV}")

## Step 1 — Fetch all metadata records from the Europeana API

The Europeana API uses cursor-based pagination. We fetch records in batches of 100 until all 9,021 have been retrieved.

In [ ]:
def fetch_all_records(collection, fonds, api_key, page_size=100):
    """Fetch all records for a given collection and fonds from the Europeana API.
    Uses cursor-based pagination to retrieve all results.
    Returns a list of raw API record dictionaries.
    """
    base_url = "https://api.europeana.eu/record/v2/search.json"
    records  = []
    cursor   = "*"

    # First request to get total count
    params = {
        "wskey":   api_key,
        "query":   f'europeana_collectionName:"{collection}" AND proxy_dc_source:"{fonds}"',
        "rows":    page_size,
        "cursor":  cursor,
        "profile": "rich",
        "media":   "true",
    }
    r = requests.get(base_url, params=params, timeout=30)
    r.raise_for_status()
    data  = r.json()
    total = data.get("totalResults", 0)
    items = data.get("items", [])
    records.extend(items)
    cursor = data.get("nextCursor")

    with tqdm(total=total, initial=len(items), unit="record", desc="Fetching") as pbar:
        while cursor:
            params["cursor"] = cursor
            r = requests.get(base_url, params=params, timeout=30)
            r.raise_for_status()
            data  = r.json()
            items = data.get("items", [])
            if not items:
                break
            records.extend(items)
            pbar.update(len(items))
            next_cursor = data.get("nextCursor")
            if not next_cursor or next_cursor == cursor:
                break
            cursor = next_cursor

    return records

print("Fetching Massafont records from Europeana...")
records = fetch_all_records(COLLECTION, FONDS, API_KEY)
print(f"\nTotal records fetched: {len(records)}")

## Step 2 — Parse records and save metadata to CSV

We extract the most useful fields from each record and save them to a CSV file. The resulting CSV has one row per photograph with the following columns:

- `europeana_id` — unique record identifier
- `title` — title in English and Catalan
- `description_en` / `description_ca` — descriptions in English and Catalan
- `creator` — always "Martí Massafont Costals"
- `date` — date of photograph (ISO format where available)
- `place` — one or more place names (semicolon-separated)
- `rights` — CC BY-NC-ND 4.0
- `image_url` — direct URL to the JPEG image
- `source_url` — URL to the record in the CRDI archive
- `europeana_url` — URL to the record on Europeana

In [ ]:
def parse_record(item):
    """Extract key fields from a raw Europeana API record."""
    return {
        "europeana_id":   item.get("id"),
        "title":          "; ".join(item.get("title", [])),
        "description_en": "; ".join(item.get("dcDescriptionLangAware", {}).get("en", [])),
        "description_ca": "; ".join(item.get("dcDescriptionLangAware", {}).get("ca", [])),
        "creator":        item.get("edmAgentLabel", [{}])[0].get("def", ""),
        "date":           item.get("edmTimespanLabel", [{}])[0].get("def", ""),
        "place":          "; ".join(
                              v for v in item.get("dctermsSpatial", [])
                              if not v.startswith("http")
                          ),
        "rights":         "; ".join(item.get("rights", [])),
        "image_url":      "; ".join(item.get("edmIsShownBy", [])),
        "source_url":     "; ".join(item.get("edmIsShownAt", [])),
        "europeana_url":  item.get("guid", ""),
    }

df = pd.DataFrame([parse_record(r) for r in records])
df.to_csv(METADATA_CSV, index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} records → {METADATA_CSV}")
df.head()

## Step 3 — Download images

Downloads all JPEG images to the output folder. Images that have already been downloaded are skipped, so this cell can be safely re-run if interrupted.

**Note:** A 0.5 second delay between requests is included to avoid overloading the CRDI server.

In [ ]:
def safe_filename(europeana_id, url):
    """Generate a safe local filename from a Europeana ID and image URL."""
    ext  = Path(url.split("?")[0]).suffix or ".jpg"
    name = europeana_id.strip("/").replace("/", "_")
    return f"{name}{ext}"

def download_images(df, output_folder, delay=0.5):
    """Download all images in the dataframe to the output folder.
    Skips images that have already been downloaded.
    """
    skipped = failed = downloaded = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Downloading"):
        url = row["image_url"]
        if pd.isna(url) or url == "":
            skipped += 1
            continue

        fname = safe_filename(row["europeana_id"], url)
        dest  = Path(output_folder) / fname

        if dest.exists():   # skip already-downloaded images
            skipped += 1
            continue

        try:
            r = requests.get(url, timeout=30, stream=True)
            r.raise_for_status()
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            downloaded += 1
        except Exception as e:
            print(f"\n  Failed {url}: {e}")
            failed += 1

        time.sleep(delay)

    print(f"\nDone — downloaded: {downloaded}, skipped: {skipped}, failed: {failed}")

download_images(df, OUTPUT_FOLDER, delay=DELAY_S)